In [ ]:
!pip install anndata

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import anndata as ad

from sklearn.neighbors import NearestNeighbors
from statsmodels.stats.multitest import multipletests

BASE = "/content"
k = 20
n_perm = 10_000
n_subsample = 50_000

def build_exact_knn_adjacency(X, k=20):
    if k >= X.shape[0]:
        raise ValueError("k must be smaller than the number of cells.")
    model = NearestNeighbors(n_neighbors=k + 1, metric="euclidean", algorithm="auto", n_jobs=-1)
    model.fit(X)
    _, indices = model.kneighbors(X)
    neighbor_indices = indices[:, 1:k + 1]
    n_cells = X.shape[0]
    rows = np.repeat(np.arange(n_cells), k)
    cols = neighbor_indices.ravel()
    values = np.ones(n_cells * k, dtype=np.int8)
    W = sp.csr_matrix((values, (rows, cols)), shape=(n_cells, n_cells))
    W.setdiag(0)
    W.eliminate_zeros()
    degree = np.asarray(W.sum(axis=1)).ravel()
    if not np.all(degree == k):
        raise RuntimeError(f"Expected exactly {k} neighbours per cell, degrees range {degree.min()}-{degree.max()}.")
    return W, neighbor_indices

def one_hot_labels(labels):
    labels = np.asarray(labels).astype(str)
    label_names, label_idx = np.unique(labels, return_inverse=True)
    n_cells = len(labels)
    n_labels = len(label_names)
    B = sp.csr_matrix(
        (np.ones(n_cells, dtype=np.int8), (np.arange(n_cells), label_idx)),
        shape=(n_cells, n_labels)
    )
    return B, label_idx, label_names

def same_label_counts(W, B, target_label_idx):
    counts_by_label = (W @ B).toarray()
    return counts_by_label[np.arange(W.shape[0]), target_label_idx]

def run_permutation_null(W, B, original_label_idx, n_perm=10_000, seed=0):
    rng = np.random.default_rng(seed)
    n_cells = B.shape[0]
    max_degree = int(np.asarray(W.sum(axis=1)).max())
    dtype = np.uint8 if max_degree <= np.iinfo(np.uint8).max else np.uint16
    null_counts = np.empty((n_cells, n_perm), dtype=dtype)
    row_numbers = np.arange(n_cells)
    for p in range(n_perm):
        permutation = rng.permutation(n_cells)
        B_perm = B[permutation, :]
        counts_by_label = (W @ B_perm).toarray()
        null_counts[:, p] = counts_by_label[row_numbers, original_label_idx]
    return null_counts

def empirical_upper_tail_pvalues(x_obs, null_counts):
    n_perm = null_counts.shape[1]
    n_as_or_more_extreme = np.sum(null_counts >= x_obs[:, None], axis=1)
    return (n_as_or_more_extreme + 1) / (n_perm + 1)

def run_rescue_pipeline(round_label, k=20, n_perm=10_000, n_subsample=50_000, seed=0):
    validation_pca_df = pd.read_csv(f"{BASE}/{round_label}_pca_validation.csv", index_col=0)
    clusters_df = pd.read_csv(f"{BASE}/{round_label}_clusters.csv").set_index("CellID")

    validation_pca_df.index = validation_pca_df.index.astype(str)
    clusters_df.index = clusters_df.index.astype(str)

    validation_pca_df = validation_pca_df.loc[~validation_pca_df.isna().any(axis=1)].copy()
    common_ids = validation_pca_df.index.intersection(clusters_df.index)
    validation_pca_df = validation_pca_df.loc[common_ids].copy()
    clusters_df = clusters_df.loc[common_ids].copy()

    if not validation_pca_df.index.equals(clusters_df.index):
        raise RuntimeError(f"{round_label}: PCA and cluster rows are not aligned.")

    print(f"\n[{round_label}] Aligned cells: {len(common_ids):,}")

    rng = np.random.default_rng(0)
    if len(validation_pca_df) > n_subsample:
        sampled_idx = rng.choice(validation_pca_df.index.to_numpy(), size=n_subsample, replace=False)
        validation_pca_df = validation_pca_df.loc[sampled_idx].copy()
        clusters_df = clusters_df.loc[sampled_idx].copy()

    print(f"[{round_label}] Using {len(validation_pca_df):,} cells")

    adata = ad.AnnData(X=np.zeros((len(validation_pca_df), 1), dtype=np.float32))
    adata.obsm["X_pca_validation"] = validation_pca_df.to_numpy(dtype=np.float32)
    adata.obs_names = validation_pca_df.index
    adata.obs["cluster"] = clusters_df["cluster"].astype(str)

    print(f"[{round_label}] Loaded {adata.n_obs} cells, {adata.obs['cluster'].nunique()} clusters.")

    W, _ = build_exact_knn_adjacency(adata.obsm["X_pca_validation"], k=k)

    labels = adata.obs["cluster"].to_numpy()
    B, label_idx, label_names = one_hot_labels(labels)

    x_obs = same_label_counts(W, B, label_idx)
    observed_fraction = x_obs / k

    cluster_sizes = np.bincount(label_idx, minlength=len(label_names))
    global_label_frequency = (cluster_sizes[label_idx] - 1) / (len(label_idx) - 1)
    fold_enrichment = np.divide(
        observed_fraction, global_label_frequency,
        out=np.full_like(observed_fraction, np.nan, dtype=float),
        where=global_label_frequency > 0
    )

    minimum_same_label_fraction = 0.50
    minimum_fold_enrichment = 2.0
    strong_enrichment = (observed_fraction >= minimum_same_label_fraction) & (fold_enrichment >= minimum_fold_enrichment)

    print(f"[{round_label}] Running permutations...")
    null_counts = run_permutation_null(W=W, B=B, original_label_idx=label_idx, n_perm=n_perm, seed=seed)
    p_vals = empirical_upper_tail_pvalues(x_obs, null_counts)
    _, q_vals, _, _ = multipletests(p_vals, alpha=0.01, method="fdr_bh")

    significant = q_vals < 0.01
    rescued = significant & strong_enrichment

    adata.obs["x_obs"] = x_obs
    adata.obs["observed_fraction"] = observed_fraction
    adata.obs["fold_enrichment"] = fold_enrichment
    adata.obs["p_value"] = p_vals
    adata.obs["q_value"] = q_vals
    adata.obs["significant"] = significant
    adata.obs["rescued"] = rescued

    print(f"[{round_label}] Rescued: {rescued.sum():,} / {adata.n_obs:,} ({100*rescued.mean():.2f}%)")

    adata.obs.to_csv(f"{BASE}/rescue_results_{round_label}.csv")
    return adata.obs["rescued"]

round1_rescued = run_rescue_pipeline("round1", k=k, n_perm=n_perm, n_subsample=n_subsample, seed=0)
round2_rescued = run_rescue_pipeline("round2", k=k, n_perm=n_perm, n_subsample=n_subsample, seed=0)

common = round1_rescued.index.intersection(round2_rescued.index)
r1 = round1_rescued.loc[common]
r2 = round2_rescued.loc[common]

print(f"\nCells common to both rounds: {len(common):,}")
print(f"Round 1 rescue %: {100*r1.mean():.2f}")
print(f"Round 2 rescue %: {100*r2.mean():.2f}")

rescued_both = (r1 & r2).sum()
rescued_r1_only = (r1 & ~r2).sum()
rescued_r2_only = (~r1 & r2).sum()
rescued_neither = (~r1 & ~r2).sum()

print(f"\nRescued in both rounds: {rescued_both:,}")
print(f"Rescued round 1 only:   {rescued_r1_only:,}")
print(f"Rescued round 2 only:   {rescued_r2_only:,}")
print(f"Rescued in neither:     {rescued_neither:,}")

jaccard = rescued_both / (rescued_both + rescued_r1_only + rescued_r2_only)
print(f"\nJaccard overlap of rescued sets: {jaccard:.3f}")

comparison_df = pd.DataFrame({"round1_rescued": r1, "round2_rescued": r2})
comparison_df.to_csv(f"{BASE}/round1_vs_round2_comparison.csv")


[round1] Aligned cells: 187,239
[round1] Using 50,000 cells
[round1] Loaded 50000 cells, 17 clusters.
[round1] Running permutations...
[round1] Rescued: 34,730 / 50,000 (69.46%)

[round2] Aligned cells: 212,226
[round2] Using 50,000 cells
[round2] Loaded 50000 cells, 13 clusters.
[round2] Running permutations...
[round2] Rescued: 34,255 / 50,000 (68.51%)

Cells common to both rounds: 11,842
Round 1 rescue %: 69.79
Round 2 rescue %: 68.98

Rescued in both rounds: 6,385
Rescued round 1 only:   1,879
Rescued round 2 only:   1,784
Rescued in neither:     1,794

Jaccard overlap of rescued sets: 0.635


In [ ]:
round1_rescued = run_rescue_pipeline("round1", k=k, n_perm=n_perm, n_subsample=n_subsample, seed=0)
round2_rescued = run_rescue_pipeline("round2", k=k, n_perm=n_perm, n_subsample=n_subsample, seed=0)


[round1] Aligned cells: 212,226
[round1] Using 50,000 cells
[round1] Loaded 50000 cells, 17 clusters.
[round1] Running permutations...
[round1] Rescued: 34,005 / 50,000 (68.01%)

[round2] Aligned cells: 212,226
[round2] Using 50,000 cells
[round2] Loaded 50000 cells, 13 clusters.
[round2] Running permutations...
[round2] Rescued: 34,255 / 50,000 (68.51%)


In [ ]:
common = round1_rescued.index.intersection(round2_rescued.index)
r1 = round1_rescued.loc[common]
r2 = round2_rescued.loc[common]

print(f"\nCells common to both rounds: {len(common):,}")
print(f"Round 1 rescue %: {100*r1.mean():.2f}")
print(f"Round 2 rescue %: {100*r2.mean():.2f}")

rescued_both = (r1 & r2).sum()
rescued_r1_only = (r1 & ~r2).sum()
rescued_r2_only = (~r1 & r2).sum()
rescued_neither = (~r1 & ~r2).sum()

print(f"\nRescued in both rounds: {rescued_both:,}")
print(f"Rescued round 1 only:   {rescued_r1_only:,}")
print(f"Rescued round 2 only:   {rescued_r2_only:,}")
print(f"Rescued in neither:     {rescued_neither:,}")

jaccard = rescued_both / (rescued_both + rescued_r1_only + rescued_r2_only)
print(f"\nJaccard overlap of rescued sets: {jaccard:.3f}")

comparison_df = pd.DataFrame({"round1_rescued": r1, "round2_rescued": r2})
comparison_df.to_csv(f"{BASE}/round1_vs_round2_comparison.csv")


Cells common to both rounds: 50,000
Round 1 rescue %: 68.01
Round 2 rescue %: 68.51

Rescued in both rounds: 26,430
Rescued round 1 only:   7,575
Rescued round 2 only:   7,825
Rescued in neither:     8,170

Jaccard overlap of rescued sets: 0.632
